In [ ]:
import math
from pathlib import Path
import numpy as np
import scipy
from scipy import linalg
from scipy.special import genlaguerre

"""
This script computes linear and quadratic Volterra kernels for lift and pitching
moment coefficients using unsteady aerodynamic data.


Outputs
-------
Dataset/Volterra/kernels_CL_CM_linear.npy : linear Volterra kernels
Dataset/Volterra/kernels_CL_CM_NL.npy     : quadratic Volterra kernels
"""

In [ ]:
def butter_filter(signal: np.ndarray, order: int, cutoff: float) -> np.ndarray:
    """
    Apply a zero phase Butterworth filter to a signal.

    Parameters
    ----------
    signal : ndarray
        Input signal to be filtered.
    order : int
        Order of the Butterworth filter.
    cutoff : float
        Normalized cutoff frequency (0 < cutoff < 1, with 1 corresponding to Nyquist frequency).

    Returns
    -------
    ndarray
        Filtered signal, same shape as input.
    """
    b, a = scipy.signal.butter(order, cutoff)
    return scipy.signal.filtfilt(b, a, signal)


def build_laguerre(tau: np.ndarray, n_terms: int, kind: str = "scipy") -> np.ndarray:
    """
    Construct a Laguerre function matrix for a given time vector.

    Parameters
    ----------
    tau : ndarray
        Time-like variable (shape: nt,).
    n_terms : int
        Number of Laguerre functions (columns) to generate.
    kind : str, optional
        Type of Laguerre functions to use ("scipy" or "dowell"), by default "scipy".

    Returns
    -------
    ndarray
        Laguerre matrix of shape (len(tau), n_terms).
    """
    if kind == "scipy":
        a = 1e-20
        alpha = -0.999
        L = np.empty((tau.size, n_terms))
        for j in range(n_terms):
            L[:, j] = genlaguerre(j, alpha)(tau) * np.exp(-a * tau)
        return L

    if kind == "dowell":
        # Dowell style
        a = 0.3
        L = np.empty((tau.size, n_terms))
        for j in range(n_terms):
            series = np.zeros_like(tau)
            for k in range(j):
                coeff = (
                    math.sqrt(2 * a)
                    * ((-1) ** k)
                    * math.factorial(j)
                    * 2 ** (j - k)
                    * (2 * a * tau) ** (j - k)
                    * np.exp(-a * tau)
                    / (math.factorial(k) * math.factorial(j - k) ** 2)
                )
                series += coeff
            L[:, j] = series
        return L


def build_convolution_matrix(alpha_hist: np.ndarray) -> np.ndarray:
    """
    Construct a lower triangular Toeplitz matrix for convolution with an input history.

    Parameters
    ----------
    alpha_hist : ndarray
        Input history vector (shape: n,).

    Returns
    -------
    ndarray
        Lower triangular Toeplitz convolution matrix (shape: n, n).
    """
    n = alpha_hist.size
    U = np.zeros((n, n))
    for k in range(n):
        U += alpha_hist[k] * np.diag(np.ones(n - k), -k)
    return U

In [ ]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

data_dir = Path("..\Dataset\CFD")
laguerre_type = "scipy"          # options: "scipy" or "dowell"
plot_flag_1_deg_response = False
plot_flag_2_deg_response_NL = False

chord = 0.4064                  # m
area = 2 * chord ** 2           # m²
x_30 = 0.12192                  # m
x_50 = 0.2032                   # m

otref = 500.0                   # 1/s   (physical time constant)
dt = 5e-5                       # s
aoA_ampl_1deg = np.pi / 180.0   # rad
aoA_ampl_2deg = 2.0             # deg (used only to build the delta)

factor = 2.0                    # multiplier for delta contribution
n_coeff = 17                    # number of Laguerre functions to retain
sample_step = 18                # corresponds to tau_sampling = 0.6

# -----------------------------------------------------------------------------
# Data loading
# -----------------------------------------------------------------------------

dataset_CL0_CM0_M_AoA = np.load(data_dir / "dataset_CL0_CM0_M_AoA.npy") # Steady state data
dataset_CL_CM_1_deg = np.load(data_dir / "dataset_CL_CM_1_deg.npy") # Unsteady data of 1 degree AoA step response
dataset_CL_CM_2_deg = np.load(data_dir / "dataset_CL_CM_2_deg.npy") # Unsteady data of 2 degree AoA step response

# Reorder to shape (cases, variables, time)
dataset_CL0_CM0_M_AoA = dataset_CL0_CM0_M_AoA.transpose(0, 2, 1)
dataset_CL_CM_1_deg = dataset_CL_CM_1_deg.transpose(0, 2, 1)
dataset_CL_CM_2_deg = dataset_CL_CM_2_deg.transpose(0, 2, 1)

Mach = dataset_CL0_CM0_M_AoA[:, 0, 0]
AoA = dataset_CL0_CM0_M_AoA[:, 1, 0]
cl0 = dataset_CL0_CM0_M_AoA[:, 2, :]
cm0 = dataset_CL0_CM0_M_AoA[:, 3, :]

# Moment reference shift
cm0 += (x_50 - x_30) * cl0 / chord

CL_1_deg = dataset_CL_CM_1_deg[:, 0, :]
CM_1_deg = dataset_CL_CM_1_deg[:, 1, :]
CL_2_deg = dataset_CL_CM_2_deg[:, 0, :]
CM_2_deg = dataset_CL_CM_2_deg[:, 1, :]

# -----------------------------------------------------------------------------
# Containers for results
# -----------------------------------------------------------------------------

kernels_CL = []
kernels_CM = []
kernels_CL_NL = []
kernels_CM_NL = []


In [ ]:
# -----------------------------------------------------------------------------
# Main loop 
# -----------------------------------------------------------------------------

for i in range(CL_1_deg.shape[0]):
    V = Mach[i] * np.sqrt(1.116 * 81.49 * 304.2128)  # Compute freestream velocity for current case

    it_all = np.arange(CL_1_deg.shape[1])  # Time indices
    cl_raw = CL_1_deg[i] - cl0[i]          # Remove steady-state CL to get unsteady component
    cm_raw = CM_1_deg[i] - cm0[i]          # Remove steady-state CM to get unsteady component
    cl_filt = butter_filter(cl_raw, 1, 0.02)   # Filter CL signal
    cm_filt = butter_filter(cm_raw, 2, 0.007)  # Filter CM signal

    time = it_all * dt                          # Physical time vector
    tau = it_all * dt * 2 * V / chord           # Non-dimensionalized time (tau)
    it_sampled = it_all[::sample_step]          # Downsample indices
    cl_sampled = cl_filt[::sample_step]         # Downsampled filtered CL
    cm_sampled = cm_filt[::sample_step]         # Downsampled filtered CM

    alpha_hist = aoA_ampl_1deg * (1.0 - np.exp(-time[it_sampled] * otref))  # Input history for 1 deg step
    tau_sampled = it_sampled * dt * 2 * V / chord                           # Downsampled tau

    L = build_laguerre(tau_sampled, n_coeff, laguerre_type)  # Laguerre matrix
    U = build_convolution_matrix(alpha_hist)                 # Convolution matrix

    uu, s, vh = linalg.svd(U @ L, full_matrices=False)       # SVD for least squares solution
    theta_cl = vh.T @ (np.diag(1 / s) @ (uu.T @ cl_sampled)) # Solve for CL kernel coefficients
    theta_cm = vh.T @ (np.diag(1 / s) @ (uu.T @ cm_sampled)) # Solve for CM kernel coefficients

    kernels_CL.append(L @ theta_cl)  # Store reconstructed linear CL kernel
    kernels_CM.append(L @ theta_cm)  # Store reconstructed linear CM kernel

    # Nonlinear delta (2 deg - scaled 1 deg)
    cl_raw_2 = CL_2_deg[i] - cl0[i]          # Remove steady-state CL for 2 deg input
    cm_raw_2 = CM_2_deg[i] - cm0[i]          # Remove steady-state CM for 2 deg input
    cl_filt_2 = butter_filter(cl_raw_2, 1, 0.02)   # Filter CL signal for 2 deg
    cm_filt_2 = butter_filter(cm_raw_2, 2, 0.007)  # Filter CM signal for 2 deg

    cl_sampled_2 = cl_filt_2[::sample_step]  # Downsampled filtered CL for 2 deg
    cm_sampled_2 = cm_filt_2[::sample_step]  # Downsampled filtered CM for 2 deg

    n_train_delta = min(len(cl_sampled), len(cl_sampled_2))  # Ensure matching lengths
    cl_delta = -cl_sampled[:n_train_delta] * factor + cl_sampled_2[:n_train_delta]  # Nonlinear CL delta
    cm_delta = -cm_sampled[:n_train_delta] * factor + cm_sampled_2[:n_train_delta]  # Nonlinear CM delta

    time_delta = np.arange(n_train_delta) * dt  # Time vector for delta
    tau_delta = np.arange(n_train_delta) * sample_step * dt * 2 * V / chord  # Tau for delta
    alpha_hist_delta = aoA_ampl_2deg * (1.0 - np.exp(-time_delta * otref))  # Input history for 2 deg step

    L_NL = build_laguerre(tau_delta, n_coeff, laguerre_type)  # Laguerre matrix for nonlinear kernel
    U2 = build_convolution_matrix(alpha_hist_delta)           # Convolution matrix for nonlinear kernel

    uu, s, vh = linalg.svd(U2 @ L_NL, full_matrices=False)    # SVD for nonlinear least squares
    theta_cl_NL = vh.T @ (np.diag(1 / s) @ (uu.T @ cl_delta)) # Solve for nonlinear CL kernel coefficients
    theta_cm_NL = vh.T @ (np.diag(1 / s) @ (uu.T @ cm_delta)) # Solve for nonlinear CM kernel coefficients

    taper = np.tanh(1.0 - 0.02 * np.linspace(0, L_NL.shape[0], L_NL.shape[0]))  # Tapering for stability
    kernels_CL_NL.append((L_NL @ theta_cl_NL) * taper)  # Store nonlinear CL kernel
    kernels_CM_NL.append((L_NL @ theta_cm_NL) * taper)  # Store nonlinear CM kernel



In [ ]:
# -----------------------------------------------------------------------------
# Convert lists to arrays and save
# -----------------------------------------------------------------------------

kernels_CL = np.array(kernels_CL)
kernels_CM = np.array(kernels_CM)
kernels_CL_NL = np.array(kernels_CL_NL)
kernels_CM_NL = np.array(kernels_CM_NL)

volterra_dir = Path("Dataset") / "Volterra"
volterra_dir.mkdir(parents=True, exist_ok=True)
np.save(volterra_dir / "kernels_CL_CM_linear.npy", np.stack((kernels_CL, kernels_CM), axis=2))
np.save(volterra_dir / "kernels_CL_CM_NL.npy", np.stack((kernels_CL_NL, kernels_CM_NL), axis=2))

print("\nKernels written to", volterra_dir)